In [1]:
# ============================================================
# G29 Reproducibility: Setup and Environment Capture
# ============================================================

# Install vLLM (pin a version if you need exact reproducibility;
# for this run we use the latest release available at execution time).
!pip install -q vllm

# Clone vLLM to record the exact commit used for benchmarking.
!git clone -q https://github.com/vllm-project/vllm.git
!cd vllm && git rev-parse HEAD > /content/vllm_commit.txt

# Capture installed vLLM version, pip freeze, and GPU details.
!vllm --version > /content/vllm_version.txt
!pip freeze > /content/requirements.txt
!nvidia-smi > /content/nvidia_smi.txt

# Download ShareGPT dataset used by vLLM's official benchmarks.
!wget -q https://huggingface.co/datasets/anon8231489123/ShareGPT_Vicuna_unfiltered/resolve/main/ShareGPT_V3_unfiltered_cleaned_split.json -O /content/sharegpt.json

import torch, subprocess, os

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)
print("vLLM commit:", open('/content/vllm_commit.txt').read().strip())
print("vLLM version:", open('/content/vllm_version.txt').read().strip())

MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
NUM_PROMPTS = 300          # keep 300 for the initial reproduction; increase later if needed
MAX_MODEL_LEN = 2048
SEED = 0

fatal: destination path 'vllm' already exists and is not an empty directory.
GPU: Tesla T4
VRAM (GB): 15.637086208
vLLM commit: 2a54f6b625f28109180b072b704c0b0d372a277d
vLLM version: 0.31.0


In [2]:
# ============================================================
# G29 Reproducibility: Throughput benchmark for block sizes 8, 16, 32
# Uses the current vLLM CLI: `vllm bench throughput`
# ============================================================

import subprocess, os

block_sizes = [8, 16, 32]
results = {}

for bs in block_sizes:
    print(f"\n=== Running throughput benchmark, block_size={bs} ===")
    cmd = [
        "vllm", "bench", "throughput",
        "--model", MODEL,
        "--dataset-name", "sharegpt",
        "--dataset-path", "/content/sharegpt.json",
        "--num-prompts", str(NUM_PROMPTS),
        "--block-size", str(bs),
        "--max-model-len", str(MAX_MODEL_LEN),
        "--seed", str(SEED),
    ]
    try:
        out = subprocess.run(cmd, capture_output=True, text=True, check=True)
        log = out.stdout + out.stderr
        tail = "\n".join(log.strip().splitlines()[-20:])
        results[bs] = tail
        print(tail)
    except subprocess.CalledProcessError as e:
        # Capture failure (e.g., block size 8 unsupported on T4)
        log = e.stdout + e.stderr
        results[bs] = f"FAILED: {e}\n{log}"
        print(f"Block size {bs} failed. See log below.\n{log}")

    # Save individual log regardless of success/failure
    with open(f"/content/throughput_block{bs}.log", "w") as f:
        f.write(results[bs])

# Print clean summary
print("\n\n===== SUMMARY (copy into report) =====")
for bs, tail in results.items():
    print(f"\n--- block_size={bs} ---\n{tail}")

print("\nLogs saved as /content/throughput_block{8,16,32}.log")


=== Running throughput benchmark, block_size=8 ===
Block size 8 failed. See log below.
(MainProcess pid=16388) INFO 10-07 14:11:47 [api_utils.py:286] non-default args: {'tokenizer': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0', 'max_model_len': 2048, 'block_size': 8, 'mm_device_do_normalize': None, 'enable_lora': None, 'reasoning_parser_plugin': '', 'logging_config': LoggingConfig(log_level='INFO', configure_logging=True, pylogging_config_file=None), 'model': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'}
(MainProcess pid=16388) INFO 10-07 14:11:47 [model.py:699] Resolved architecture: LlamaForCausalLM
(MainProcess pid=16388) WARNING 10-07 14:11:47 [model.py:2389] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
(MainProcess pid=16388) WARNING 10-07 14:11:47 [model.py:2442] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=16388) INFO 10-07 14:11:47 [model.py:2109] Using max model len 2048
(MainProcess pid=163

In [3]:
# ============================================================
# G29 Reproducibility: Azure LLM Inference Trace analysis
# Produces length distributions, arrival burstiness, and request class split.
# ============================================================

import pandas as pd
import numpy as np
import requests

# Direct CSV link from Azure Public Dataset
CSV_URL = "https://raw.githubusercontent.com/Azure/AzurePublicDataset/master/data/AzureLLMInferenceTrace_conv.csv"

df = pd.read_csv(CSV_URL)
print("Columns:", df.columns.tolist())
print(df.head())

# Identify context and generated token columns
ctx_col = [c for c in df.columns if "Context" in c or "Input" in c][0]
gen_col = [c for c in df.columns if "Generated" in c or "Output" in c][0]

print("\nContext tokens:\n", df[ctx_col].describe())
print("\nGenerated tokens:\n", df[gen_col].describe())

# Arrival burstiness
ts_col = "TIMESTAMP"
df[ts_col] = pd.to_datetime(df[ts_col])
arrivals_per_sec = df.set_index(ts_col).resample("1S").size()
print("\nArrivals/sec stats:\n", arrivals_per_sec.describe())
burst_factor = arrivals_per_sec.max() / arrivals_per_sec.mean()
print("Burst factor (max/mean):", burst_factor)

# Bucket into short/medium/long
df['req_class'] = pd.cut(df[ctx_col], bins=[0, 128, 512, 1e9], labels=['short', 'medium', 'long'])
print("\nRequest class distribution:\n", df['req_class'].value_counts(normalize=True))

# Save summary for report
summary = {
    "context_tokens_mean": df[ctx_col].mean(),
    "context_tokens_p90": df[ctx_col].quantile(0.9),
    "generated_tokens_mean": df[gen_col].mean(),
    "generated_tokens_p90": df[gen_col].quantile(0.9),
    "arrivals_per_sec_mean": arrivals_per_sec.mean(),
    "burst_factor": burst_factor,
}

print("\n===== SUMMARY FOR REPORT =====")
with open("/content/azure_trace_summary.txt", "w") as f:
    for k, v in summary.items():
        line = f"{k}: {v:.2f}"
        print(line)
        f.write(line + "\n")

# Save enriched CSV for later simulator use
df.to_csv("/content/azure_trace_with_classes.csv", index=False)
print("\nSaved /content/azure_trace_with_classes.csv and /content/azure_trace_summary.txt")

Columns: ['TIMESTAMP', 'ContextTokens', 'GeneratedTokens']
                     TIMESTAMP  ContextTokens  GeneratedTokens
0  2023-11-16 18:15:46.6805900            374               44
1  2023-11-16 18:15:50.9951690            396              109
2  2023-11-16 18:15:51.2224670            879               55
3  2023-11-16 18:15:51.3910170             91               16
4  2023-11-16 18:15:52.5732450             91               16

Context tokens:
 count    19366.000000
mean      1154.697408
std       1108.822564
min          2.000000
25%        396.000000
50%       1020.000000
75%       1189.000000
max      14050.000000
Name: ContextTokens, dtype: float64

Generated tokens:
 count    19366.000000
mean       211.125942
std        162.870478
min          7.000000
25%         85.000000
50%        129.000000
75%        395.000000
max       1000.000000
Name: GeneratedTokens, dtype: float64

Arrivals/sec stats:
 count    3503.000000
mean        5.528404
std         2.710415
min         0.

/tmp/ipykernel_16113/3715615797.py:27: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  arrivals_per_sec = df.set_index(ts_col).resample("1S").size()



Saved /content/azure_trace_with_classes.csv and /content/azure_trace_summary.txt


In [4]:
# ============================================================
# G29 Reproducibility: Collect all evidence and download as zip
# ============================================================

import os
from google.colab import files

outputs = [
    "/content/vllm_commit.txt",
    "/content/vllm_version.txt",
    "/content/requirements.txt",
    "/content/nvidia_smi.txt",
    "/content/throughput_block8.log",
    "/content/throughput_block16.log",
    "/content/throughput_block32.log",
    "/content/azure_trace_summary.txt",
    "/content/azure_trace_with_classes.csv",
]

existing = [f for f in outputs if os.path.exists(f)]
missing = [f for f in outputs if not os.path.exists(f)]

if missing:
    print("Not found (skipping):", missing)

# Create zip
!zip -j /content/g29_reproducibility_package.zip {' '.join(existing)}

print("\nDownloading zip...")
files.download("/content/g29_reproducibility_package.zip")

  adding: vllm_commit.txt (stored 0%)
  adding: vllm_version.txt (stored 0%)
  adding: requirements.txt (deflated 58%)
  adding: nvidia_smi.txt (deflated 76%)
  adding: throughput_block8.log (deflated 83%)
  adding: throughput_block16.log (deflated 84%)
  adding: throughput_block32.log (deflated 84%)
  adding: azure_trace_summary.txt (deflated 34%)
  adding: azure_trace_with_classes.csv (deflated 76%)



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>